# 高性能网络服务器面试速记（C++/Linux/网络）

本笔记基于本仓库实现（Reactor 多线程、epoll ET、非阻塞 IO、写缓冲、跨线程安全、MySQL 异步队列等）整理，覆盖高频考点与易错点，附关键代码/思路。

- 覆盖方向：网络 IO、Linux 基础、并发模型、锁与内存、数据库与安全、性能优化与压测。
- 目标：面试前快速过一遍，临场答题有框架、有细节、有示例。


## 1. epoll：ET vs LT 与正确读写循环

- 区别：
  - LT（Level-Triggered）水平触发：只要缓冲区有数据就会持续触发，常配合阻塞式读取直到没有可读数据。
  - ET（Edge-Triggered）边沿触发：状态变化才触发（从无到有），必须使用非阻塞并在一次回调中循环读/写直到 `EAGAIN`。
- 正确读取（ET + 非阻塞）的关键：
  1) socket 设为非阻塞
  2) 回调里用 `while(read(...)>0)` 循环
  3) `read==-1 && (errno==EAGAIN||EWOULDBLOCK)` 作为“本次读完”的结束条件
- 正确写入（写缓冲 + EPOLLOUT）：
  - 直接 `write`，若返回 `EAGAIN`/未写完：把剩余数据入写缓冲，注册 `EPOLLOUT`，在可写事件回调继续发送，直至全部发送完；然后取消 `EPOLLOUT` 保留 `EPOLLIN`。
- 常见错误：回调只读一次；忘记非阻塞；不处理 `EAGAIN`；写缓冲生命周期管理不当。


## 2. Reactor 多线程模型与连接分配（代码级详解）

### 架构总览
```
主线程（mainReactor）
  └─ Acceptor：监听 ServerSocket，accept 新连接
           ↓
      根据 fd 哈希分配到某个 subReactor（N 个线程，N = CPU 核数）
           ↓
subReactor 1, 2, ..., N（并行运行）
  └─ 每个 subReactor 有独立的 EventLoop + Epoll + 连接集合
```

### 关键代码实现

#### 1. Server 初始化（创建主从 Reactor）
```cpp
// Server.cpp
Server::Server(EventLoop* loop): mainReactor(loop), acceptor(nullptr) {
    // 创建 Acceptor，绑定到主 Reactor
    acceptor = new Acceptor(mainReactor);  // mainReactor 负责监听
    
    // 设置 accept 回调
    acceptor->setNewConnectionCallback(std::bind(&Server::newConnection, this, _1));
    
    // 创建 N 个子 Reactor（N = CPU 核数）
    int size = std::thread::hardware_concurrency();  // 例如：8 核 → 8 个 subReactor
    thpool = new ThreadPool(size);
    
    for (int i = 0; i < size; ++i) {
        subReactors.push_back(new EventLoop());  // 每个 subReactor 有独立 EventLoop
    }
    
    // 启动所有 subReactor 线程
    for (int i = 0; i < size; ++i) {
        thpool->add(std::bind(&EventLoop::loop, subReactors[i]));  
        // 每个线程运行一个 EventLoop::loop()
    }
}
```
- 要点：主 Reactor 处理 listenfd；创建 N 个 subReactor；每个运行独立线程与 EventLoop。

#### 2. Acceptor 监听与 accept 循环
```cpp
// Acceptor.cpp
Acceptor::Acceptor(EventLoop* _loop) : loop(_loop) {
    sock = new Socket();  // ServerSocket
    sock->sbind(inetAddr);  // bind(127.0.0.1:8080)
    sock->slisten();  // listen()
    sock->setnonblocking();  // 非阻塞 accept
    
    // 将 listenfd 注册到主 Reactor 的 Epoll
    Channel* svrChannel = new Channel(loop, sock->getFd());
    svrChannel->SetReadCallback(std::bind(&Acceptor::acceptConnection, this));
    svrChannel->enableReading();  // 关注 EPOLLIN
}

void Acceptor::acceptConnection() {
    // accept 循环：一次 accept 所有就绪连接（ET 模式）
    while (true) {
        InetAddress* client_addr = new InetAddress();
        int cliFd = sock->saccept(client_addr);  // 非阻塞 accept
        
        if (cliFd == -1) {
            if (errno == EAGAIN) break;  // 没有更多连接，退出循环
            perror("accept");
            break;
        }
        
        // 将新连接传给 Server 处理
        newConnectionCallback(new Socket(cliFd));  // 回调 Server::newConnection
    }
}
```
- 要点：listenfd 在主 Reactor 监听；非阻塞，一次 accept 多条连接；回调到 `Server::newConnection`。

#### 3. 连接分配与创建（关键优化）
```cpp
// Server.cpp
void Server::newConnection(Socket *serv_sock) {
    int fd = serv_sock->getFd();
    
    // 负载均衡：用 fd 取模分配到某个 subReactor
    int random = fd % subReactors.size();  // 例如：fd=123, N=8 → random=3
    EventLoop* subLoop = subReactors[random];  // 分配到 subReactor #3
    
    // 【关键优化】将 Connection 创建投递到 subReactor 线程执行
    subLoop->runInLoop([this, serv_sock, fd, subLoop]() {
        // 在 subReactor 线程中创建 Connection 和 Channel
        Connection *conn = new Connection(subLoop, serv_sock);
        conn->setDeleteConnectionCallback(std::bind(&Server::deleteConnection, this, _1));
        
        // 将连接加入统一管理（加锁保护）
        {
            std::lock_guard<std::mutex> guard(mutex_);
            connections[fd] = conn;
        }
        Manager::getInstance().append(fd, conn);
    });
}
```
- 要点：按 fd 哈希分配；Connection 创建投递到目标 subReactor；连接与 Channel 在同一线程。

#### 4. EventLoop 跨线程唤醒（eventfd）
```cpp
// EventLoop.cpp
EventLoop::EventLoop() {
    // 创建 eventfd，用于唤醒阻塞的 epoll_wait
    wakeupFd = eventfd(0, EFD_NONBLOCK | EFD_CLOEXEC);
    wakeupChannel = new Channel(this, wakeupFd);
    wakeupChannel->SetReadCallback(std::bind(&EventLoop::handleWakeupRead, this));
    wakeupChannel->enableReading();  // 监听 eventfd 可读
}

void EventLoop::queueInLoop(const std::function<void()>& cb) {
    // 加锁将任务放入队列
    {
        std::lock_guard<std::mutex> lock(pendingMutex);
        pendingFunctors.push_back(cb);
    }
    wakeup();  // 写 eventfd 唤醒 EventLoop
}

void EventLoop::wakeup() {
    uint64_t one = 1;
    write(wakeupFd, &one, sizeof(one));  // 触发 epoll_wait 返回
}

void EventLoop::loop() {
    while (!quit) {
        std::vector<Channel*> chs = epoll->poll();  // epoll_wait 阻塞等待
        for (auto* ch : chs) {
            ch->handleEvent();  // 处理 IO 事件
        }
        doPendingFunctors();  // 执行跨线程投递的任务
    }
}

void EventLoop::doPendingFunctors() {
    std::vector<std::function<void()>> functors;
    // 快速交换，减少锁持有时间
    {
        std::lock_guard<std::mutex> lock(pendingMutex);
        functors.swap(pendingFunctors);
    }
    for (auto& f : functors) f();
}
```
- 要点：主线程通过 eventfd 唤醒目标 subReactor；被唤醒线程批量执行投递任务；锁内放任务，交换后批量执行。

#### 5. 连接删除与资源回收
```cpp
// Server.cpp
void Server::deleteConnection(int sock) {
    Connection* conn = nullptr;
    {
        // 锁内只做查找和移除，不做耗时操作
        std::lock_guard<std::mutex> guard(mutex_);
        auto it = connections.find(sock);
        if (it != connections.end()) {
            conn = it->second;
            connections.erase(it);  // 从映射移除
        }
    }
    // 锁外 delete Connection，避免析构阻塞其他线程
    if (conn) {
        delete conn;  // 析构自动关闭 fd、返回 Buffer、删除 Channel
    }
}
```
- 要点：锁内只做必要维护；锁外析构；Connection 析构释放资源。

### 优势总结
- 并行 IO：每个 subReactor 独立 epoll；CPU 使用均衡。
- 减少竞争：按 fd 分配；连接创建在同一线程；锁外析构；广播快速复制。
- 高效唤醒：eventfd 轻量；跨线程安全；批量执行任务。

### 面试话术
- “采用 Reactor 多线程，主 Reactor 负责 accept，按 fd 哈希分配，投递到对应 subReactor 线程创建 Connection 并注册 Channel，避免跨线程操作。”



## 3. TCP 粘包/拆包、消息边界与协议设计

- 粘包原因：TCP 是字节流，无消息边界；内核缓冲合并；Nagle/Delayed ACK 等。
- 常见解决：
  - 定长消息头（长度 + 类型），后跟 payload 按长度读取。
  - 分隔符（如本项目使用 ASCII 3 `\x03` 分隔 name 与 payload）。
  - TLV/变长编码（varint）。
- 面试要点：
  - 正确处理半包/多包：循环读取、缓存拼接、状态机解析。
  - 指出分隔符方案的转义问题（或限制字符集）。



## 4. 写缓冲与 EPOLLOUT 正确姿势（关键伪代码）

```cpp
// 伪代码：发送数据（优先直写，写不完入缓冲并注册 EPOLLOUT）
ssize_t n = ::write(fd, data + sent, total - sent);
if (n == -1) {
  if (errno == EAGAIN || errno == EWOULDBLOCK) {
    writeBuffer.append(data + sent, total - sent);
    enableWriting(fd); // 注册 EPOLLOUT
    return;
  }
  closeConn(fd);
  return;
}
sent += n;
if (sent < total) {
  writeBuffer.append(data + sent, total - sent);
  enableWriting(fd);
}

// EPOLLOUT 回调：尽量把缓冲发空，发空后取消 EPOLLOUT
while (!writeBuffer.empty()) {
  ssize_t n = ::write(fd, writeBuffer.data(), writeBuffer.size());
  if (n == -1 && (errno == EAGAIN || errno == EWOULDBLOCK)) return; // 等下次可写
  if (n <= 0) { closeConn(fd); return; }
  writeBuffer.consume(n);
}

disableWriting(fd); // 取消 EPOLLOUT（但保留 EPOLLIN）
```

- 注意事项：
  - 不可一直保持 EPOLLOUT（会造成忙等回调）；只在有未写完数据时注册。
  - 线程安全：写操作应在所属 Reactor 线程执行；跨线程用 `queueInLoop` 投递。


## 5. 锁与并发优化：shared_mutex、thread_local、锁外回调

### 5.1 shared_mutex 优化（多读并发）

#### 场景：Manager 的读写锁
```cpp
// Manager.h
class Manager {
    std::shared_mutex rwMutex_;  // shared_mutex 替代 mutex
    std::unordered_map<int, Connection*> connections_;
    
public:
    void append(int fd, Connection* conn) {
        std::unique_lock<std::shared_mutex> lock(rwMutex_);  // 写锁
        connections_[fd] = conn;
    }
    
    void broadcast(int senderFd, std::function<void(Connection*)> fn) {
        // 读取操作：先复制列表（共享锁，允许多个读并发）
        std::vector<Connection*> targets;
        {
            std::shared_lock<std::shared_mutex> lock(rwMutex_);  // 共享读锁
            for (auto& [fd, conn] : connections_) {
                if (fd != senderFd) targets.push_back(conn);
            }
        }  // 锁已释放
        
        // 锁外执行回调，避免阻塞其他操作
        for (auto* conn : targets) fn(conn);
    }
};
```

**优势**：
- 读多写少：多个读操作并发；仅写操作互斥。
- 性能：相比 `mutex`，读吞吐提升。

---

### 5.2 thread_local 缓存（避免全局锁）

#### 概念说明
**没有 thread_local 的传统做法**：
```cpp
class BufferPool {
    std::vector<Buffer*> pool_;
    std::mutex mtx_;  // 全局锁
    
public:
    Buffer* getBuffer() {
        std::lock_guard<std::mutex> lock(mtx_);  // 每次都要加锁！
        if (!pool_.empty()) {
            auto buf = pool_.back();
            pool_.pop_back();
            return buf;
        }
        return new Buffer();
    }
    
    void returnBuffer(Buffer* buf) {
        std::lock_guard<std::mutex> lock(mtx_);  // 每次都要加锁！
        pool_.push_back(buf);
    }
};
```
- 问题：频繁加锁导致竞争。

**使用 thread_local 优化**：
```cpp
// BufferPool.cpp
// 【关键】每个线程有自己的独立缓存，无需加锁
thread_local std::vector<std::shared_ptr<Buffer>> thread_local_cache;

BufferPool::getBuffer() {
    // 【无锁】优先从当前线程的本地缓存获取
    if (!thread_local_cache.empty()) {
        auto buf = thread_local_cache.back();
        thread_local_cache.pop_back();
        return buf;  // 80-90% 的操作走这里，完全无锁！
    }
    
    // 本地缓存为空，才从全局池获取（需要加锁）
    {
        std::lock_guard<std::mutex> lock(mtx_);
        if (!pool_.empty()) {
            auto buf = pool_.back();
            pool_.pop_back();
            return buf;
        }
    }
    
    return std::make_shared<Buffer>();  // 创建新的
}

BufferPool::returnBuffer(std::shared_ptr<Buffer> buf) {
    buf->clear();
    
    // 【无锁】优先放入本地缓存
    if (thread_local_cache.size() < THREAD_LOCAL_CACHE_SIZE * 2) {
        thread_local_cache.push_back(buf);
    } else {
        // 本地缓存满了，回收到全局池（需要加锁）
        std::lock_guard<std::mutex> lock(mtx_);
        pool_.push_back(buf);
    }
}
```

#### 工作流程示意
```
线程1（subReactor 1）      线程2（subReactor 2）      线程3（subReactor 3）
┌─────────────────┐       ┌─────────────────┐       ┌─────────────────┐
│ thread_local    │       │ thread_local    │       │ thread_local    │
│ cache: [B1,B2]  │       │ cache: [B3,B4]  │       │ cache: [B5,B6]  │
│                 │       │                 │       │                 │
│ getBuffer()     │       │ getBuffer()     │       │ getBuffer()     │
│ └─> 返回 B1     │       │ └─> 返回 B3     │       │ └─> 返回 B5     │
│     (无锁!)     │       │     (无锁!)     │       │     (无锁!)     │
└─────────────────┘       └─────────────────┘       └─────────────────┘
         ▲                        ▲                        ▲
         │                        │                        │
         └────────────────────────┴────────────────────────┘
                                  │
                      ┌───────────┴───────────┐
                      │   全局池（全局锁）      │
                      │  pool: [B7,B8,...]   │
                      └───────────────────────┘
```

#### 性能对比
- 无 thread_local：每次 `getBuffer/returnBuffer` 都加锁 → 竞争重
- 有 thread_local：约 80–90% 从本地缓存走 → 更低延迟与更高吞吐

**面试话术**：
- "`thread_local` 让每线程有独立缓存；连接复用 Buffer 多数走本地，本地满才访问全局池。"

---

### 5.3 锁外回调（减少持锁时间）

```cpp
void Manager::broadcast(int senderFd, std::function<void(Connection*)> fn) {
    std::vector<Connection*> targets;
    
    // 【阶段1】持锁复制列表（短时间）
    {
        std::shared_lock<std::shared_mutex> lock(rwMutex_);
        for (auto& [fd, conn] : connections_) {
            if (fd != senderFd) targets.push_back(conn);
        }
    }  // 锁已释放
    
    // 【阶段2】锁外执行回调（可能耗时）
    for (auto* conn : targets) {
        conn->postSend(...);  // 可能涉及 write、分配、回调
    }
}
```

**对比错误做法**：
```cpp
// 错误：持锁回调
void broadcast(..., fn) {
    std::shared_lock lock(rwMutex_);
    for (auto& conn : connections_) {
        fn(conn);  // 回调可能很慢，长时间持锁！
    }
}  // 锁才释放
```

---

### 5.4 连接创建/删除的锁外优化

```cpp
// Server.cpp
void Server::deleteConnection(int sock) {
    Connection* conn = nullptr;
    {
        // 锁内只做查找和移除（快速）
        std::lock_guard<std::mutex> guard(mutex_);
        auto it = connections.find(sock);
        if (it != connections.end()) {
            conn = it->second;
            connections.erase(it);  // 从映射移除
        }
    }  // 锁已释放
    
    // 锁外 delete Connection（可能耗时）
    if (conn) {
        delete conn;  // 析构、关闭 fd、返回 Buffer
    }
}
```

**面试话术**：
- "锁内只做必要维护；耗时操作放锁外，减少竞争与延迟。"

---

### 总结对比

| 优化 | 传统做法 | 优化后 | 提升 |
|------|---------|--------|------|
| **shared_mutex** | 全局读写用 `mutex` | 多读共享锁，写独占锁 | 读吞吐提升 |
| **thread_local** | 全局池 + 全局锁 | 每线程独立缓存 | 约 80–90% 无锁 |
| **锁外回调** | 持锁执行回调 | 快速复制列表后锁外执行 | 持锁时间减少 |
| **锁外析构** | 锁内析构 | 锁内移除，锁外析构 | 中断更快 |

**记忆点**：读多共享锁；本地缓存降锁；锁外做慢操作；锁内只维护结构。



## 6. 跨线程安全与唤醒：eventfd + runInLoop/queueInLoop（深入解析）

### 问题场景

**业务场景**：主 Reactor 收到新连接，需要将 Connection 创建投递到某个 subReactor 线程。

**核心问题**：
- subReactor 线程在 `epoll_wait` 阻塞
- 主 Reactor 需告知目标 subReactor 有新任务
- 如何安全唤醒与调度

### 错误做法（为什么不能用）

#### 错误1：直接跨线程操作 epoll（崩溃/未定义行为）

```cpp
// 错误做法1：直接跨线程操作 epoll
void forceWakeup() {
    EventLoop* targetLoop = subReactors[i];
    // 在另一个线程强行修改 epoll？
    // ❌ epoll 不是线程安全的！
    epoll_ctl(targetLoop->epoll, EPOLL_CTL_ADD, ...);  // 崩溃/未定义行为
}
```

**为什么崩溃/未定义行为？** 以下原因：

**原因1：epoll 的内核数据结构不是线程安全的**
```cpp
// Epoll.cpp 实际调用
void Epoll::updateChannel(Channel* channel) {
    struct epoll_event ev;
    ev.data.ptr = channel;  // 存储 Channel 指针
    
    // 这一步在跨线程时会竞态！
    epoll_ctl(epfd, EPOLL_CTL_ADD, fd, &ev);
    channel->setInEpoll();  // 设置状态标志
}
```

**竞态场景**：
```
主线程（mainReactor）          subReactor 线程
      │                              │
      │ epoll_ctl(ADD, fd1)          │
      ├──────────────────────────────>│ 内核修改 epoll 数据结构
      │                              │
      │                              │ epoll_wait() 返回
      │                              │ 正在遍历 events 数组
      │                              │
      │ epoll_ctl(ADD, fd2) ←───────┼─ 同一时刻内核也在修改 epoll
      ├──────────────────────────────>│ 竞态条件！
      │                              │
      │ 结果：内核数据结构不一致     │ 结果：events 数组可能损坏
      │ ❌ 崩溃或未定义行为          │ ❌ 读到垃圾数据
```

**原因2：Channel 状态管理不是原子操作**
```cpp
// Connection.cpp - Channel 创建和注册
Connection::Connection(EventLoop* loop, Socket* sock) {
    if (loop->isInLoopThread()) {
        // 在所属线程：创建 Channel 并设置状态
        channel = new Channel(loop, sock->getFd());
        channel->SetReadCallback(...);
        channel->enableReading();  // 调用 epoll_ctl
    }
}

// Channel.cpp
void Channel::enableReading() {
    events |= EPOLLIN | EPOLLET;
    loop->updateChannel(this);  // 触发 epoll_ctl
}

// Epoll.cpp
void Epoll::updateChannel(Channel* channel) {
    if (!channel->getInEpoll()) {  // 状态判断
        epoll_ctl(epfd, EPOLL_CTL_ADD, fd, &ev);
        channel->setInEpoll();  // 设置状态
    }
}
```

**跨线程竞态**：
```
主线程                          subReactor 线程
   │                               │
   │ 检查 getInEpoll() → false    │
   ├──────────────────────────────>│
   │                               │ epoll_ctl(ADD) 执行
   │ 检查 getInEpoll() → false    │ setInEpoll() 设置
   ├──────────────────────────────>│ 重复 ADD！
   │ epoll_ctl(ADD) ←─────────────┼─ Channel 已添加
   │                               │
   │ ❌ EEXIST 错误或崩溃          │ ❌ 状态不一致
```

**原因3：epoll_wait 和 epoll_ctl 并发修改**
```cpp
void EventLoop::loop() {
    while (!quit) {
        std::vector<Channel*> chs = epoll->poll();  // 内部是 epoll_wait
        
        // 如果此时另一个线程调用 epoll_ctl 修改 epoll
        // events 数组可能处于不一致状态
        for (auto* ch : chs) {
            ch->handleEvent();
        }
    }
}
```

**竞态时序**：
```
时间线：
T1: subReactor 调用 epoll_wait(epfd, events, MAX_EVENTS, -1)
    └─> 内核开始填充 events 数组（假设填充到 event[5]）
    
T2: 主线程调用 epoll_ctl(epfd, EPOLL_CTL_ADD, newFd, ...)
    └─> 内核修改 epoll 的红黑树（添加新 fd）
    
T3: subReactor 的 epoll_wait 继续填充 events[6...]
    └─> ❌ events 数组包含新旧混合状态
    
T4: subReactor 读取 events 数组
    └─> ❌ 读到部分初始化的 events，指向无效 Channel
    └─> ❌ 访问野指针 → 崩溃
```

// 错误做法2：轮询/忙等
void badWakeup() {
    targetLoop->pendingTasks.push(task);
    // 如何通知？轮询？
    while (!targetLoop->finished) sleep(1);  // ❌ CPU 浪费
}

// 错误做法3：信号
void signalWakeup() {
    targetLoop->pendingTasks.push(task);
    pthread_kill(targetThread, SIGUSR1);  // ❌ 信号处理复杂，容易丢失
}
```

**问题总结**：
- 直接操作 epoll：不安全（竞态、悬垂指针、数据损坏）
- 轮询：浪费 CPU
- 信号：复杂易丢

**面试要点**：
- epoll 非线程安全；跨线程 `epoll_ctl` 会与 `epoll_wait` 竞态，导致 events 不一致/野指针/崩溃
- Channel 的 `check-modify-set` 非原子，跨线程会重复 ADD 或状态不一致
- eventfd 方案可避免：同一线程内操作 epoll，唤醒仅写入 fd，不修改 epoll 数据结构

---

### 正确方案：eventfd + queueInLoop

#### 1. eventfd 基础

**eventfd 是什么**：
```cpp
int wakeupFd = eventfd(0, EFD_NONBLOCK | EFD_CLOEXEC);
```
- 专用于事件通知的文件描述符
- 可读可写；非阻塞；计数累积
- 写入计数后 epoll 返回可读；读后清空计数

**工作机制**：
```
写端（任意线程）             读端（subReactor 线程）
  │                           │
  │  write(1)                 │
  ├──────────────────────────>│ 触发 epoll_wait 返回
  │                           │  Channel::handleEvent()
  │                           │  读数据 → 事件计数清零
  │                           │
```

#### 2. 完整流程代码实现

**步骤1：初始化 eventfd**
```cpp
// EventLoop.cpp
EventLoop::EventLoop() {
    threadId = std::this_thread::get_id();
    
    // 创建 eventfd，用于跨线程唤醒
    wakeupFd = eventfd(0, EFD_NONBLOCK | EFD_CLOEXEC);
    // EFD_NONBLOCK: 非阻塞
    // EFD_CLOEXEC: 继承时关闭
    
    // 将 eventfd 注册到自己的 epoll
    wakeupChannel = new Channel(this, wakeupFd);
    wakeupChannel->SetReadCallback(std::bind(&EventLoop::handleWakeupRead, this));
    wakeupChannel->enableReading();  // 关注 EPOLLIN
}
```

**步骤2：主 Reactor 投递任务**
```cpp
// Server.cpp
void Server::newConnection(Socket* serv_sock) {
    int fd = serv_sock->getFd();
    EventLoop* subLoop = subReactors[fd % subReactors.size()];
    
    // 【关键】将任务投递到 subReactor 线程
    subLoop->runInLoop([this, serv_sock, fd, subLoop]() {
        Connection* conn = new Connection(subLoop, serv_sock);
        // ... 后续处理
    });
}
```

**步骤3：任务投递逻辑**
```cpp
// EventLoop.cpp
void EventLoop::runInLoop(const std::function<void()>& cb) {
    if (isInLoopThread()) {
        // 如果已经在 EventLoop 线程，直接执行
        cb();
    } else {
        // 如果不在，投递到任务队列
        queueInLoop(cb);
    }
}

void EventLoop::queueInLoop(const std::function<void()>& cb) {
    // 加锁将任务放入队列
    {
        std::lock_guard<std::mutex> lock(pendingMutex);
        pendingFunctors.push_back(cb);
    }
    
    // 【关键】写 eventfd 唤醒阻塞的 epoll_wait
    wakeup();  
}

void EventLoop::wakeup() {
    uint64_t one = 1;
    ssize_t n = write(wakeupFd, &one, sizeof(one));
    // 写入一个 1，触发 eventfd 可读
}
```

**步骤4：subReactor 处理唤醒**
```cpp
// EventLoop.cpp
void EventLoop::loop() {
    while (!quit) {
        // epoll_wait 阻塞，等待事件
        std::vector<Channel*> chs = epoll->poll();
        
        for (auto* ch : chs) {
            ch->handleEvent();  // 处理 IO 事件
        }
        
        // 【关键】处理跨线程投递的任务
        doPendingFunctors();  
    }
}

void EventLoop::doPendingFunctors() {
    std::vector<std::function<void()>> functors;
    {
        // 快速交换，减少锁持有时间
        std::lock_guard<std::mutex> lock(pendingMutex);
        functors.swap(pendingFunctors);  // pendingFunctors 被清空
    }
    
    // 锁外批量执行任务
    for (auto& f : functors) {
        f();
    }
}

void EventLoop::handleWakeupRead() {
    uint64_t one;
    ssize_t n = read(wakeupFd, &one, sizeof(one));
    // 读取并清零 eventfd 计数
    // 不需要处理数据，只是唤醒 epoll_wait
}
```

#### 3. 完整时序图

```
主线程（mainReactor）          subReactor 线程
      │                              │
      │  1. 新连接到达               │
      │  2. 选择 subLoop             │
      │  3. subLoop->runInLoop(...)  │
      │                              │
      ├─────────────────────────────>│ 4. queueInLoop()
      │                              │    - 加锁 push 任务
      │                              │    - wakeup()
      │                              │
      │  5. write(wakeupFd, 1)       │
      ├─────────────────────────────>│ 6. eventfd 触发
      │                              │
      │                              │ 7. epoll_wait 返回
      │                              │ 8. Channel::handleEvent()
      │                              │ 9. handleWakeupRead()
      │                              │10. doPendingFunctors()
      │                              │    - swap 任务队列
      │                              │    - 批量执行任务
      │                              │
      │                              │11. 创建 Connection
      │                              │12. 注册 Channel
      │                              │13. 加入 Manager
      │                              │
```

#### 4. 关键设计点

**为什么用 eventfd 而不是 pipe/socketpair？**
```cpp
// pipe 创建两个 fd，eventfd 只有一个 fd
int pipefd[2];
pipe(pipefd);  // pipefd[0] 读，pipefd[1] 写

// eventfd 更轻量，开销更小
int fd = eventfd(0, EFD_NONBLOCK);  // 只有一个 fd
```
- 开销更小；接口单一；适合唤醒

**为什么批量执行任务？**
```cpp
// 错误做法：逐个执行
for (auto& f : pendingFunctors) {
    std::lock_guard lock(mutex_);
    f();
}  // 持有锁时间过长

// 正确做法：批量 swap 后执行
std::vector functors;
{  // 只在这个大括号内加锁
    std::lock_guard lock(mutex_);
    functors.swap(pendingFunctors);  // 快进快出
}
for (auto& f : functors) f();  // 锁外执行
```
- 减少持锁时间

**为什么需要 isInLoopThread() 判断？**
```cpp
void runInLoop(const std::function<void()>& cb) {
    if (isInLoopThread()) {
        cb();  // 同线程直接执行
    } else {
        queueInLoop(cb);  // 跨线程投递
    }
}
```
- 同线程直接执行；跨线程才唤醒

---

### 实战应用示例

**主 Reactor 投递连接创建**：
```cpp
void Server::newConnection(Socket* serv_sock) {
    EventLoop* subLoop = subReactors[fd % subReactors.size()];
    
    // 投递到 subReactor 线程创建
    subLoop->runInLoop([this, serv_sock, fd, subLoop]() {
        Connection* conn = new Connection(subLoop, serv_sock);
        // 在 subReactor 线程中创建和注册，避免跨线程竞争
    });
}
```

**跨线程发送消息**：
```cpp
void Connection::postSend(std::string data) {
    // 确保在所属 EventLoop 线程执行
    loop->queueInLoop([this, data]() {
        send(data);  // 在正确的线程发送
    });
}
```

---

### 面试话术

**问**："为什么用 eventfd 而不是信号？"
- 答："eventfd 开销更低；一对一更清晰；非阻塞；计数累积；作为 fd 自然接入 epoll，信号易丢。"

**问**："为什么不能直接跨线程改 epoll？"
- 答："不线程安全。正确路径：投递任务到队列，写 eventfd，在目标线程执行任务并注册。"

**问**："如何保证任务顺序？"
- 答："同一连接固定在同一 subReactor，顺序由 TCP 与事件循环保证。"


## 7. 数据库与安全：异步队列 + SQL 注入防护

- 异步队列：IO 线程将 SQL 文本入队，由 N 个 worker 并行执行，避免 IO 线程阻塞数据库。
- 注入防护：对所有用户输入 `mysql_real_escape_string` 转义；更佳方案是“预编译 + 绑定参数”。
- 线程模型：worker 拥有各自连接池/连接，避免跨线程共享同一连接。


## 8. TCP 细节：Nagle、Delayed ACK、TIME_WAIT/CLOSE_WAIT

- Nagle：合并小包，降低报文数；低延迟场景常 `TCP_NODELAY` 关闭。
- Delayed ACK：延迟确认以减少 ACK 包；与 Nagle 同时开启会放大延迟。
- TIME_WAIT：主动关闭方进入，持续 2MSL；可通过复用端口（`SO_REUSEADDR`）与合理连接复用降低影响。
- CLOSE_WAIT：对端关闭，我方未 `close`；排查应用层未释放资源的逻辑问题。


## 9. 内存与资源：对象池、零拷贝、泄漏排查

- 对象池/BufferPool：高频复用，减少分配/释放与碎片；结合 `thread_local` 降锁开销。
- 零拷贝方向：`sendfile/mmap/splice`；项目侧可减少不必要内存复制（共享指针、预分配、就地拼接）。
- 泄漏排查：`valgrind/memcheck`、`asan/lsan`；多线程竞态：`tsan`。
- 资源释放：连接析构在锁外执行；RAII 封装避免遗漏。


## 10. 压测与指标：QPS、P99、吞吐、成功率

- 关键指标：QPS、成功率、平均/中位/分位延迟（P95/P99）、吞吐（MB/s）。
- 压测策略：冷/热启动；升压/稳压；单机/分布式；有/无数据库；不同消息大小与并发数。
- 常见瓶颈：锁冲突、上下文切换、内核参数、内存复制、磁盘/数据库、GC/大页缺失。


## 11. 代码题速练：最小 epoll 服务器（骨架）

```cpp
int ep = epoll_create1(0);
int listenfd = create_listen_fd(port); set_nonblock(listenfd);
add_read_event(ep, listenfd);

std::vector<Conn> conns;
while (true) {
  epoll_event evs[1024];
  int n = epoll_wait(ep, evs, 1024, -1);
  for (int i=0;i<n;++i) {
    int fd = evs[i].data.fd;
    if (fd == listenfd) {
      while (true) {
        int cfd = accept4(listenfd, nullptr, nullptr, SOCK_NONBLOCK);
        if (cfd == -1) { if (errno==EAGAIN) break; /*handle*/ }
        add_read_event(ep, cfd, ET);
      }
    } else if (evs[i].events & EPOLLIN) {
      while (true) {
        ssize_t m = read(fd, buf, sizeof(buf));
        if (m > 0) { /* 解析/入业务缓冲 */ }
        else if (m == 0) { close(fd); break; }
        else if (errno==EAGAIN) { break; }
        else { close(fd); break; }
      }
    } else if (evs[i].events & EPOLLOUT) {
      // 从写缓冲发送，发空则取消 EPOLLOUT
    }
  }
}
```

- 评分点：非阻塞、ET 循环、accept 循环、读写错误处理、写缓冲与事件切换。


## 12. 手撕：组装协议帧（name + \x03 + payload）

```cpp
std::string assemble(const std::string& name, const std::string& payload) {
  std::string out; out.reserve(name.size()+1+payload.size());
  out.append(name); out.push_back('\x03'); out.append(payload);
  return out;
}

bool parse(const std::string& in, std::string& name, std::string& payload) {
  auto p = in.find('\x03');
  if (p == std::string::npos) return false;
  name = in.substr(0, p);
  payload = in.substr(p+1);
  return true;
}
```

- 面试提示：说明分隔符方案的转义问题与限制，并给出“长度前缀”的可选设计。


## 13. 快速问答清单（背诵版）

- 为什么 ET 必须非阻塞并循环读到 EAGAIN？
  - 因为边沿只在状态变化触发，必须一次性把缓冲读干净，否则数据留在内核里但不会再次触发。
- 写缓冲为什么要取消 EPOLLOUT？
  - 发空就撤销，避免持续触发造成 CPU 忙等；下次有数据再注册。
- 主/从 Reactor 的优势？
  - 利用多核并行 IO，降低竞争；连接创建、事件注册在所属线程完成，减少跨线程同步。
- 粘包如何处理？
  - 长度前缀、分隔符、TLV；配合状态机与缓存。
- Nagle 与 Delayed ACK 影响？
  - 小包延迟；低延迟业务可关 Nagle（`TCP_NODELAY`），评估与 Delayed ACK 的耦合。
- TIME_WAIT 太多怎么办？
  - 合理复用（`SO_REUSEADDR`）、连接复用、反压策略，评估是否频繁主动关闭。
- 数据库如何避免阻塞 IO 线程？
  - SQL 文本入队，worker 并行执行；IO 线程只投递，不等待结果。
- 注入如何防护？
  - 转义或预编译；输入校验与权限最小化。
- 如何降低锁竞争？
  - `shared_mutex` 多读并发、`thread_local` 缓存、锁外回调、减少临界区。
- 发现和定位内存泄漏？
  - asan/lsan、valgrind；加统计、对齐对象生命周期。


## 14. 架构类高频问答（系统设计视角）

- 问：为什么选择 Reactor 而不是 Proactor？
  - 答：Linux 下传统 Proactor 需要内核/库层完成异步读写回调，接口复杂且落地成本高；Reactor + 非阻塞 + epoll ET 已足够高效、易控、可与线程模型自然组合。高吞吐与低延迟场景里，Reactor 的可预期性更高。

- 问：单机扩展策略？
  - 答：主/从 Reactor 多线程并行 IO、连接创建并行化、读写分离（读多路径优化）、`shared_mutex` 降低读竞争、`thread_local` 对象池消除全局锁、写缓冲 + EPOLLOUT 降低系统调用次数。

- 问：水平扩展（多机）如何做？
  - 答：前置 L4/七层代理（如 LVS/Nginx/Envoy）做负载均衡；实例内保持无状态或将状态外部化（Redis/一致性哈希分片）；会话粘性或 Token 携带最小状态；广播/房间模型用消息总线（Kafka/NATS）或分片路由。

- 问：限流与回压如何设计？
  - 答：多层限流（连接数、每 IP、新建速率、消息速率、SQL 入队速率）；队列水位触发丢弃/降级；写缓冲设置上限并拒绝继续写；利用 epoll 的背压信号（不可写时不继续 push 数据）。

- 问：拥塞/洪泛（流量激增）怎么保护？
  - 答：接受队列与线程池排队上限；过载丢弃/拒绝新连接（Shed Load）；熔断下游（数据库/缓存）并快速失败；降级非关键功能（历史持久化、统计）。

- 问：负载均衡策略选择？
  - 答：轮询、连接数最少、加权、EWMA 延迟；聊天室类长连接可用“房间一致性哈希”以减少跨节点广播成本；跨 AZ/Region 需健康检查与故障转移策略。

- 问：状态管理与一致性？
  - 答：尽量无状态；必须有状态时外部化到 Redis/DB，并保证幂等与去重；跨线程/跨节点的用户在线表使用版本号或乐观锁；最终一致优先，强一致需牺牲可用性或吞吐。

- 问：消息顺序与重复处理？
  - 答：单连接内顺序由 TCP 保证；跨节点广播可加单调递增序列或 Lamport 时钟；消费端保证幂等（基于消息 ID 去重）。

- 问：数据库瓶颈如何治理？
  - 答：异步队列 + 多 worker 并行；分库分表、热点键缓存、写合并、批量提交；慢查询索引与执行计划；必要时引入 CQRS（写入与查询分离）。

- 问：缓存策略与一致性？
  - 答：读多场景用 Redis/Memcached；失效策略：超时 + 主动失效；写路径采用先写 DB 后删缓存或写穿；热点 Key 做分片与本地 LRU 缓存。

- 问：观测性与可运维性？
  - 答：结构化日志（JSON）、指标（Prometheus/OpenTelemetry）、分布式追踪（TraceID 上下文穿透）；开机/热更新/优雅关闭（停止接收、等待在途、超时强杀）。

- 问：故障恢复与自愈？
  - 答：进程崩溃由系统级 Supervisor/容器编排（systemd/K8s）拉起；健康检查 + 灰度 + 回滚；限流熔断避免级联故障；DB/消息队列多副本与自动重试（幂等）。

- 问：安全与多租户？
  - 答：传输加密（TLS/WS over TLS）；鉴权（Token/HMAC），限权（RBAC）；输入校验与 SQL 注入防护；多租户隔离（命名空间/租户 ID），配额与审计。

- 问：零停机发布与配置热更新？
  - 答：双进程/蓝绿/金丝雀发布；配置中心（YAML/ENV）+ SIGHUP/热加载；连接与线程池大小可按水位自动调节。

- 问：内核与网络参数调优？
  - 答：提升 `somaxconn`、`tcp_max_syn_backlog`、`netdev_max_backlog`；关闭/调整 Nagle、增加 `rmem/wmem`；reuseport 提高 accept 并行；合理的 epoll wait 超时与线程绑核。

- 问：如何做端到端压测与容量预估？
  - 答：分阶段（无 DB、带 DB、带缓存）、分流量层级（P50→P99）、记录资源利用率（CPU/内存/网络/磁盘/连接数），基于拐点与 SLO 反推单机容量与扩容因子。

- 问：为何采用分隔符协议而非长度前缀？
  - 答：实现简单、便于调试；但需处理转义/字符集；若需要二进制/变长帧与强鲁棒，建议长度前缀或 TLV。

- 问：如何最小代价接入 Web？
  - 答：保持 TCP 协议不变，增加 WebSocket 代理（本项目 `tools/ws_proxy.py`）与简易网页；后续可原生集成 WS 或 HTTP/2 gRPC 网关。



## 15. 项目实现中的实际问题与优化（实战复盘）

### 问题1：ET 模式事件丢失（影响 QPS 和稳定性）
- 现象：部分消息丢失。
- 原因：在回调里只读一次，导致数据留在内核缓冲但不再触发。
- 解决：读循环直到 `EAGAIN`；socket 非阻塞；正确区分“异常/正常结束/未读完”。
- 效果：QPS 和可靠性提升。

```cpp
// 错误示范：只读一次
ssize_t n = ::read(fd, buf, sizeof(buf));
if (n > 0) process(buf); // 可能丢失数据！

// 正确示范：循环读
while (true) {
  ssize_t n = ::read(fd, buf, sizeof(buf));
  if (n > 0) { process(buf); }
  else if (n == -1 && (errno == EAGAIN || errno == EWOULDBLOCK)) { break; }
  else { handle_error(); break; }
}
```

### 问题2：double free 和资源泄漏
- 现象：偶发崩溃，内存占用上升。
- 原因：跨线程析构与 `close` 的竞态；缓存边界溢出导致栈破坏。
- 解决：锁外析构；RAII 结合对象池；连接与 Channel 生命周期对齐。
- 效果：长时间运行更稳定；泄漏降至 0。

### 问题3：写缓冲未实现，部分丢失
- 现象：大消息、网速慢时失败。
- 原因：只做直写，遇 `EAGAIN` 丢失。
- 解决：未写完入缓冲；注册/取消 EPOLLOUT；在可写回调继续发送。
- 效果：全场景可靠发送；吞吐与用户体验提升。

### 问题4：连接创建成为瓶颈
- 现象：高并发时 QPS 上不去，主线程 CPU 高。
- 原因：主线程串行创建 `Connection`。
- 解决：创建投递到 subReactor；在同线程直接注册 Channel；无锁无系统调用。
- 效果：百万级连接建立更高效；串行瓶颈消除。

### 问题5：Manager 连接管理锁竞争严重
- 现象：锁争夺导致性能下降。
- 原因：读多写少仍用全局锁。
- 解决：`shared_mutex` 允许多读并发；广播接口锁外回调。
- 效果：读吞吐提升；广播时其他连接不受长时间锁阻塞。

### 问题6：BufferPool 全局对象池锁开销大
- 现象：高频分配时占用高。
- 原因：全局锁竞争。
- 解决：`thread_local` 每线程缓存；池满才走全局锁。
- 效果：80-90% 取还走本地；竞争显著下降。

### 问题7：MySQL 阻塞 IO 线程
- 现象：高并发下响应明显变慢。
- 原因：同步执行 SQL。
- 解决：异步缓冲队列 + 多 worker 并行；IO 线程只入队不等待。
- 效果：IO 零阻塞；吞吐大幅提升；可复用到其他阻塞资源。

### 问题8：字符串复制与堆分配导致停顿
- 现象：小消息也有抖动。
- 原因：小字符串走堆；频繁重分配。
- 解决：栈数组用于小字符串；`reserve` 预分配；`shared_ptr` 减少拷贝。
- 效果：抖动明显下降；内存片段压力减轻。

### 压测方法与实际测试数据

**测试环境**：WSL2 on Windows，8 核 CPU，本机回环。
- 工具：`benchmark`（多线程压测、延迟分布、实时统计）
- 命令示例：`./benchmark -c 50 -r 10 -t 30`（50 并发、每秒 10 条、30 秒）

**小规模实测数据（本机）**：
- 压测场景：100 并发客户端，每客户端每秒 20 条消息，持续 60 秒。
- 实际结果：
  - QPS：约 1000–1100（消息 <100 字节）
  - 平均延迟：<0.1 ms
  - 延迟分布：100% 在 0–10 ms 内
  - 发送吞吐：约 0.03 MB/s
  - 接收吞吐：约 1.64 MB/s
  - 成功率：100%（本机回环）

**约束说明（对面试官）**：
- 本机回环延迟极低，无网络 RTT。
- 未模拟跨网络延迟、丢包与拥塞。
- 消息小，未覆盖大消息与长连接老化。
- 未考虑生产级并发与复杂边界条件。

**预估扩展（理论/算法层面）**：
- 更大并发（1000+）：预期受限于 CPU 与内存带宽；可通过分片/集群扩展。
- 生产部署：需补观测、限流、熔断、容灾与分层降级。

### 其他措施与落地效果
- 异步 DB 队列：入队失败时丢弃或降级（避免接口挂住）。
- SQL 注入防护：`mysql_real_escape_string` 转义；必要时改预编译。
- 连接生命周期与 Channel 注册/注销对齐；避免悬垂引用与复用 FD 冲突。
- 线程安全调度：`runInLoop/queueInLoop` + `eventfd` 减少 epoll 抖动。
- 写缓冲上限：防内存爆炸与队列挤压。
- 复用标准库 `shared_mutex` 替代自研读写锁；减少并发 bug。
- 错误处理：统一错误码、结构化日志、监控告警；提升可观测性。

### 经验
- 先理解原理（ET/LT、非阻塞、事件循环）再写代码。
- 用 `tsan/asan/valgrind` 与工具链做线程与内存验证。
- 核对关键路径：读/写/accept 循环、事件切换、生命周期管理。
- 分阶段压测并对比优化前后，优先消除串行瓶颈和锁竞争。
- 高并发优先正确性与可用性，再优化吞吐和时延。



## 16. 面试话术：如何诚实又专业地展示项目

**被问“QPS 多少？实际跑过吗？”**
- 若有实测：
  - 答：“我在本机用 benchmark 做了小规模压测。以 8 核 CPU 为例，单消息约 1KB，得到 QPS 约 8k–10k。
- 但这里有明显局限：
  - 本机回环延迟极低；
  - 未模拟跨网络延迟；
  - 未覆盖大消息与长连接老化；
  - 未考虑生产级并发与边界情况。
- 因此我更多关注架构与优化思路：主从 Reactor 并行 IO、读循环与 EPOLLOUT、`shared_mutex` 与 `thread_local`、异步 DB 队列。
- 这些改动在小规模压测里都有可见的提升，放大到生产需要更完整的环境与更严格的压力验证。”
- 若未压测：
  - 答：“我没做过大规模压测，但实现了完整的压测工具 `benchmark`，并记录了 QPS、成功率、延迟分布与吞吐统计。请允许我补一次实据。

- 我可以这样说：
  - 基于实现与优化，我预估单机小消息（1KB）可到 5k–10k QPS，取决于 CPU 与网卡带宽。
  - 但我更想强调的是架构：主从 Reactor 并行 IO、读循环与 EPOLLOUT、锁优化、异步数据库等，这些改动在小规模验证里有可见提升。
  - 若您需要更精确数据，我可以尽快做一次压测并提供报告。
  - 生产环境还需考虑网络延迟、消息大小分布、DB 与缓存延迟、限流与熔断。”

**被问“和开源项目相比呢？”**
- 答：“我不会对标 Nginx/Redis。这个项目更多是学习与落地并进。我实现了从零到完整的 Reactor、非阻塞 IO、跨线程调度与异步数据库等，踩过坑、做过优化，对底层原理更清楚。实际项目会用成熟库，但学习与落地两手抓更扎实。”

**被问“若让你上线，你会怎么做？”**
- 答：“先补短板：
  - 完善 MySQL 不可用时的降级（本地队列/异步落盘）与观测/追踪/灰度/限流/熔断/优雅关闭，并做运维与容灾与持续压测。”


## 17. 面试常见连环问与拆招

### 连环问1：并发、锁、上下文切换
- 问：“为什么不用多进程？”
  - 答：“进程切换重、通信复杂、共享状态成本高。多线程共享内存，通信快，本模型以事件驱动为主，切换少。”
- 追问：“你的线程如何划分？”
  - 答：“主从 Reactor：主 1 个负责 accept，子 N 个（按 CPU 核）负责 IO；另有 worker 并行执行阻塞任务（如数据库）。职责分离，扩展性好。”
- 追问：“怎么减少上下文切换？”
  - 答：“事件驱动而非忙轮询；同线程内读写不切换；`thread_local` 缓存减少跨线程访问；合理分配任务到所属线程。”

### 连环问2：epoll、信号、非阻塞
- 问：“epoll 为什么比 select/poll 快？”
  - 答：“内核维护就绪列表，返回即有事件；O(1) 加入/移除；支持边缘触发。select/poll 需遍历全部。”
- 追问：“边缘与水平选哪个？”
  - 答：“ET 适合高吞吐：触发少、系统调用少。LT 易错但好调试。我用 ET+非阻塞+读循环。”
- 追问：“非阻塞会不停轮询吗？”
  - 答：“不会。没有事件时 epoll_wait 阻塞；只在事件到来时唤醒，事件处理完毕返回继续等待。”

### 连环问3：内存、泄漏、性能
- 问：“如何减少内存分配？”
  - 答：“对象池复用 Buffer；`thread_local` 缓存避免全局锁；小字符串用栈；大对象预分配；智能指针减少拷贝。”
- 追问：“泄漏如何避免？”
  - 答：“RAII 自动管理；锁外析构避免阻塞；进程内生命周期可控；valgrind/ASAN/LSAN 定期验证。”
- 追问：“大并发下内存会爆吗？”
  - 答：“按连接数预估；池化限制；用阈值拒绝；外部化会话到 Redis。我的压测场景未爆。”

### 连环问4：负载、扩展、容错
- 问：“单机能撑多少？”
  - 答：“本机约 1k QPS；生产受限于硬件。可线性扩展到数十万连接。”
- 追问：“如何水平扩展？”
  - 答：“LB 分发；实例无状态；会话外部化；Redis 缓存；MySQL 读写分离；消息队列解耦。”
- 追问：“有状态怎么办？”
  - 答：“定时器外部化；一致性哈希分片；读写分离；最终一致优先；版本号/时钟防冲突。”

### 连环问5：设计、选择、权衡
- 问：“为什么不选现成库？”
  - 答：“学习目的，更清楚原理、可控性与调优；实际项目会选成熟的。”
- 追问：“哪些设计后来改过？”
  - 答：“锁从全局改 `shared_mutex`；连接创建从主线程改并行；DML 从同步改异步；写缓冲从丢失改可靠发送。”
- 追问：“还有什么没做？”
  - 答：“超时清理、业务限流、观测与追踪、优雅重启、多租户与资源隔离、云原生。”


## 18. 代码细节拷问速记

### 拷问1：读循环边界处理
```cpp
while (true) {
  ssize_t n = read(fd, buf, sizeof(buf));
  if (n > 0) { /* process */ }
  else if (n == 0) { close(fd); break; }  // EOF
  else if (errno == EAGAIN || errno == EWOULDBLOCK) { break; }  // 读完
  else { close(fd); break; }  // 真错误
}
```
- 要点：区分“EOF/读完/错误”，及时关 fd，避免死循环。

### 拷问2：accept 循环
```cpp
while (true) {
  int cfd = accept4(listenfd, nullptr, nullptr, SOCK_NONBLOCK);
  if (cfd == -1) {
    if (errno == EAGAIN || errno == EWOULDBLOCK) break;
    perror("accept"); continue;
  }
  handle_new_conn(cfd);
}
```
- 要点：LT 多次 accept 耗尽队列；ET 用循环处理全量就绪；设非阻塞。

### 拷问3：写缓冲生命周期
```cpp
class Connection {
  Buffer* writeBuffer;  // 生命周期对齐 Connection
public:
  void postSend(data) {
    // 先尝试直写
    if (write(fd, data) == -1 && errno == EAGAIN) {
      writeBuffer->append(data);
      enableWriting();  // 等 EPOLLOUT
    }
  }
  void onWritable() {
    while (!writeBuffer->empty()) {
      ssize_t n = write(fd, writeBuffer->data(), writeBuffer->size());
      if (n > 0) writeBuffer->consume(n);
      else if (n == -1 && errno == EAGAIN) return;
      else { close(fd); return; }
    }
    disableWriting();
  }
};
```
- 要点：写不完入缓冲；发空取消 EPOLLOUT；避免重注册与忙等。

### 拷问4：线程安全的任务队列
```cpp
class EventLoop {
  std::vector<std::function<void()>> pendingTasks;
  int wakeupFd;  // eventfd
public:
  void queueInLoop(std::function<void()> fn) {
    {
      std::lock_guard<std::mutex> lock(mutex_);
      pendingTasks.push_back(fn);
    }
    if (!isInLoopThread()) wakeup();  // 写入 eventfd 唤醒
  }
  void doPendingTasks() {
    std::vector<std::function<void()>> tasks;
    {
      std::lock_guard<std::mutex> lock(mutex_);
      tasks.swap(pendingTasks);
    }
    for (auto& task : tasks) task();
  }
};
```
- 要点：保护队列；eventfd 安全唤醒；在本线程批量执行减少临界区时长。

### 拷问5：Manager 读写锁与广播
```cpp
class Manager {
  std::shared_mutex rwMutex;
  std::unordered_map<int, Connection*> conns;
public:
  void broadcast(int senderFd, std::function<void(Connection*)> fn) {
    // 先复制列表（短锁）
    std::vector<Connection*> targets;
    {
      std::shared_lock<std::shared_mutex> lock(rwMutex);
      for (auto& [fd, conn] : conns) {
        if (fd != senderFd) targets.push_back(conn);
      }
    }
    // 锁外执行回调（避免阻塞）
    for (auto* conn : targets) fn(conn);
  }
};
```
- 要点：`shared_mutex` 多读并发；快速复制列表；锁外执行回调减少持锁时长。


## 19. 最后检查清单（面试前 5 分钟）

✅ **基础概念**
- [ ] epoll ET/LT 区别与正确用法
- [ ] Reactor vs Proactor 选择
- [ ] 读循环/写缓冲/EPOLLOUT 生命周期
- [ ] TCP 粘包与协议设计
- [ ] TIME_WAIT/CLOSE_WAIT/CLOSE/Nagle/Delayed ACK

✅ **并发模型**
- [ ] 主从 Reactor + 连接分配
- [ ] eventfd + queueInLoop 跨线程唤醒
- [ ] shared_mutex + 锁外回调降竞争
- [ ] thread_local 本地缓存
- [ ] 连接创建并行化

✅ **实战优化**
- [ ] ET 事件丢失问题与修复
- [ ] 写缓冲与 EPOLLOUT 完整流程
- [ ] MySQL 异步队列 + worker 并行
- [ ] 对象池 + 内存优化
- [ ] 实际测试数据与约束

✅ **架构扩展**
- [ ] 水平扩展与负载均衡
- [ ] 限流回压与拥塞保护
- [ ] 状态管理与一致性
- [ ] 观测与容错
- [ ] 上线的补短路径

✅ **话术准备**
- [ ] 诚实表述实测数据与限制
- [ ] 强调架构与优化思路
- [ ] 清楚与开源项目的差异
- [ ] 上线前的补短项

✅ **代码细节**
- [ ] 能讲出关键路径（读/写/accept/广播）
- [ ] 清楚锁/内存/生命周期边界
- [ ] 能用伪代码描述实现
- [ ] 能回答连环追问

**核心记忆点**：
1. ET 必须非阻塞 + 读循环，否则丢事件
2. 写不完入缓冲 + EPOLLOUT，发空即取消
3. 锁外回调 + eventfd 唤醒保证跨线程安全
4. 实测数据诚实，重点展示架构与优化思路
5. 上线前补短板：观测、限流、容错、灰度


## 20. 零拷贝技术与异常处理/资源回收详解

### 零拷贝数据传输的实现

**面试官问“你的零拷贝是如何实现的？”**

**回答策略**：先说明内核态真正的零拷贝（`sendfile/mmap/splice`），再说明项目内的优化（减少拷贝）。

#### 内核态零拷贝（真正的零拷贝）
```cpp
// 方案1：sendfile（适合文件发送）
ssize_t sendfile(int out_fd, int in_fd, off_t* offset, size_t count);
// 内核直接从文件页缓冲读取 → Socket 发送，无需经过用户态

// 方案2：mmap + write（适合大数据共享）
void* ptr = mmap(nullptr, fileSize, PROT_READ, MAP_PRIVATE, fd, 0);
write(clientFd, ptr, fileSize);  // 内核优化映射写入

// 方案3：splice（适合管道/套接字间）
ssize_t splice(int fd_in, loff_t* off_in, int fd_out, loff_t* off_out, size_t len, unsigned int flags);
```
- 适用场景：文件传输、静态资源服务。
- 本项目未使用（聊天室以内存消息为主）。

#### 项目内的零拷贝优化（减少拷贝）
```cpp
// 1. Buffer 共享指针传递，避免数据复制
class Buffer {
  std::shared_ptr<std::string> data_;  // 引用计数
public:
  Buffer* getReadBuffer() { return shared_from_this(); }  // 不拷贝内容
};

// 2. 跨线程传递用 shared_ptr，避免拷贝
void broadcast(int senderFd, std::string message) {
  auto sharedMsg = std::make_shared<std::string>(std::move(message));
  for (auto& conn : connections) {
    conn->postSend(sharedMsg);  // 传指针，不拷贝内容
  }
}

// 3. 预分配内存，减少扩容复制
void Connection::postSend(const std::string& data) {
  std::string out;
  out.reserve(name_.size() + 1 + data.size());  // 预分配，避免扩容拷贝
  out.append(name_);
  out.push_back('\x03');
  out.append(data);  // 一次性组装
}

// 4. 就地构造，避免临时对象
struct Message {
  std::string sender;
  std::string content;
  Message(std::string s, std::string c) : sender(std::move(s)), content(std::move(c)) {}
};
```
- 要点：减少不必要复制；用引用传递；预分配与移动语义；共享指针传递复杂对象。

**面试话术**：
- “项目内把数据拷贝从多层降到最少，主要通过共享指针、预分配与移动语义。真正的零拷贝（sendfile 等）用于文件传输，聊天场景不适用，重点在处理内存消息时减少复制。”

### 异常处理机制

**面试官问“你的异常处理怎么做的？”**

```cpp
// 统一错误码枚举
enum class ServerError {
  OK = 0,
  SOCKET_ERROR,
  BIND_ERROR,
  LISTEN_ERROR,
  ACCEPT_ERROR,
  READ_ERROR,
  WRITE_ERROR,
  MYSQL_ERROR,
  CONNECTION_CLOSED
};

// 错误处理封装
#define CHECK_SYS_CALL(ret, error_code) \
  if ((ret) == -1) { \
    log_error(strerror(errno), error_code); \
    return error_code; \
  }

// read 时的异常处理
ssize_t n = read(fd, buf, sizeof(buf));
if (n == -1) {
  if (errno == EINTR) continue;  // 信号中断，重试
  else if (errno == EAGAIN || errno == EWOULDBLOCK) break;  // 正常结束
  else {  // 真错误
    log_error("read failed", errno);
    close(fd);
    Manager::getInstance().remove(fd);
    return ServerError::READ_ERROR;
  }
}
if (n == 0) {  // EOF
  log_info("client disconnected");
  close(fd);
  return ServerError::CONNECTION_CLOSED;
}

// MySQL 操作异常处理
void MySQLManager::executeQuery(const std::string& sql) {
  MYSQL* conn = getConnection();
  if (!conn) {
    log_error("MySQL connection failed");
    return;
  }
  if (mysql_query(conn, sql)) {
    log_error("mysql_query failed", mysql_error(conn));
    return;
  }
  // 使用 mysql_store_result/mysql_free_result 管理结果
}
```
- 要点：统一错误码；分类处理（EINTR/EAGAIN/真错/EOF）；结构化日志与统计；资源与连接自动清理。

### 连接资源自动回收

**面试官问“连接资源如何自动回收？避免泄漏和 double free？”**

```cpp
// 1. RAII 封装 Connection 生命周期
class Connection {
  int fd_;
  Channel* channel_;
  Buffer* readBuffer_;
  Buffer* writeBuffer_;
public:
  ~Connection() {  // 析构自动回收
    if (channel_) { delete channel_; channel_ = nullptr; }
    if (readBuffer_) { BufferPool::getInstance().returnBuffer(readBuffer_); }
    if (writeBuffer_) { BufferPool::getInstance().returnBuffer(writeBuffer_); }
    if (fd_ >= 0) { close(fd_); fd_ = -1; }
  }
  
  void setDeleteCallback(std::function<void(int)> cb) {
    deleteCallback_ = cb;
  }
  void onError() {
    // 出错时调用回调，让 Server 从 Manager 移除
    deleteCallback_(fd_);
  }
};

// 2. Server 统一管理生命周期
class Server {
  std::mutex mutex_;
  std::unordered_map<int, Connection*> connections_;
  
  void deleteConnection(int fd) {
    Connection* conn = nullptr;
    {
      std::lock_guard<std::mutex> lock(mutex_);
      auto it = connections_.find(fd);
      if (it != connections_.end()) {
        conn = it->second;
        connections_.erase(it);  // 先从映射移除
      }
    }
    // 锁外 delete，避免析构阻塞其他操作
    if (conn) delete conn;
  }
  
  void newConnection(Socket* sock) {
    Connection* conn = new Connection(loop_, sock);
    conn->setDeleteCallback([this](int fd) { deleteConnection(fd); });
    conn->setErrorCallback([this](int fd) { deleteConnection(fd); });
    
    std::lock_guard<std::mutex> lock(mutex_);
    connections_[fd] = conn;  // 纳入统一管理
  }
};

// 3. BufferPool 对象池复用
class BufferPool {
  std::vector<Buffer*> freeBuffers_;
  std::mutex mutex_;
public:
  Buffer* getBuffer() {
    std::lock_guard<std::mutex> lock(mutex_);
    if (!freeBuffers_.empty()) {
      Buffer* buf = freeBuffers_.back();
      freeBuffers_.pop_back();
      return buf;
    }
    return new Buffer();
  }
  
  void returnBuffer(Buffer* buf) {
    if (!buf) return;
    buf->clear();  // 重置状态
    std::lock_guard<std::mutex> lock(mutex_);
    freeBuffers_.push_back(buf);  // 回收到池
  }
};

// 4. MySQL 结果集自动释放
class MySQLResultGuard {
  MYSQL_RES* result_;
public:
  MySQLResultGuard(MYSQL_RES* res) : result_(res) {}
  ~MySQLResultGuard() {
    if (result_) mysql_free_result(result_);
  }
  MYSQL_RES* get() { return result_; }
};

// 使用
MYSQL_RES* res = mysql_store_result(conn);
MySQLResultGuard guard(res);  // 自动释放
```
- 要点：
  - RAII 自动清理 fd/Buffer/Channel。
  - 统一管理连接，锁外析构，生命周期对齐。
  - 对象池复用，避免频繁分配/释放。
  - Guard 保护资源，避免 double free/泄漏与悬垂引用。

**面试话术**：
- “RAII 封装 Connection，析构时关闭 fd、返回 Buffer、删除 Channel；Server 统一管理映射并锁外析构；对象池复用减少开销；必要时用 Guard 保护。”

**避免的问题**：
- double free：仅 Server 管理；锁外统一析构；关闭 fd 后标记无效。
- 泄漏：RAII/池化与结构化日志/统计；定期 ASAN/Valgrind 验证。
- 悬垂引用：Manager 先移除映射；锁外析构；指针/引用不再复用。
